# Overfitting, Underfitting & Regularization Lab

A machine learning model must learn generalizable patterns rather than memorizing random training noise. This lab demonstrates the **bias-variance tradeoff** through polynomial curve fitting, inspects the U-shaped test error curve, and contrasts coefficient shrinkage across **L1 (Lasso)**, **L2 (Ridge)**, and **Elastic Net** regularization.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import mean_squared_error

np.random.seed(42)
np.set_printoptions(precision=4, suppress=True)

## 1. Polynomial Complexity and the Bias-Variance Curve

Fit polynomials of degree $d \in [1, 10]$ on 15 noisy sine-wave training samples and evaluate against 50 clean test points.

In [ ]:
# Synthetic data from sin(x)
X_train = np.random.uniform(0, 2*np.pi, 15).reshape(-1, 1)
y_train = np.sin(X_train).ravel() + np.random.normal(0, 0.2, 15)

X_test = np.linspace(0, 2*np.pi, 50).reshape(-1, 1)
y_test = np.sin(X_test).ravel()

degrees = [1, 2, 3, 5, 8, 12]
print(f"{'Degree':<8} {'Train MSE':<14} {'Test MSE':<14} {'Model Status'}")
print("-" * 52)

for deg in degrees:
    poly = PolynomialFeatures(degree=deg, include_bias=False)
    X_tr_p = poly.fit_transform(X_train)
    X_te_p = poly.transform(X_test)
    
    m = LinearRegression().fit(X_tr_p, y_train)
    tr_err = mean_squared_error(y_train, m.predict(X_tr_p))
    te_err = mean_squared_error(y_test, m.predict(X_te_p))
    
    status = "Underfitting" if deg <= 1 else ("Balanced (Sweet Spot)" if deg <= 4 else "Overfitting!")
    print(f"{deg:<8} {tr_err:<14.4f} {te_err:<14.4f} {status}")

## 2. Coefficient Shrinkage: L1 (Lasso) vs. L2 (Ridge) vs. Elastic Net

Observe how L1 sets uninformative weights to exact zero (feature selection) while L2 shrinks all weights proportionally.

In [ ]:
# Synthetic high-dimensional features where only 2 of 5 are informative
X_feat = np.random.randn(30, 5)
true_w = np.array([2.5, -1.8, 0.0, 0.0, 0.0])
y_feat = X_feat @ true_w + np.random.normal(0, 0.1, 30)

m_ols = LinearRegression().fit(X_feat, y_feat)
m_ridge = Ridge(alpha=1.0).fit(X_feat, y_feat)
m_lasso = Lasso(alpha=0.2).fit(X_feat, y_feat)
m_enet = ElasticNet(alpha=0.2, l1_ratio=0.5).fit(X_feat, y_feat)

print(f"{'Model':<16} {'w0':<8} {'w1':<8} {'w2':<8} {'w3':<8} {'w4':<8} {'Zero Weights'}")
print("-" * 68)
for name, model in [("OLS (No Reg)", m_ols), ("Ridge (L2)", m_ridge), ("Lasso (L1)", m_lasso), ("ElasticNet", m_enet)]:
    zeros = np.sum(np.isclose(model.coef_, 0, atol=1e-3))
    w_str = " ".join([f"{w:<7.3f}" for w in model.coef_])
    print(f"{name:<16} {w_str} {zeros} / 5")

print("\nTakeaway: Lasso drives noisy weights (w2, w3, w4) to exactly 0.000, performing feature selection!")

## 3. Sweeping Regularization Strength $\alpha$ ($\lambda$)

Evaluate Ridge regression across $\alpha \in [0.001, 100.0]$ to trace the transition from overfitting to underfitting.

In [ ]:
poly_deg = 8
poly_8 = PolynomialFeatures(degree=poly_deg, include_bias=False)
X_tr_8 = poly_8.fit_transform(X_train)
X_te_8 = poly_8.transform(X_test)

alphas = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]
print(f"{'Alpha':<10} {'Train Error':<14} {'Test Error':<14} {'Interpretation'}")
print("-" * 54)

for a in alphas:
    ridge = Ridge(alpha=a).fit(X_tr_8, y_train)
    tr_e = mean_squared_error(y_train, ridge.predict(X_tr_8))
    te_e = mean_squared_error(y_test, ridge.predict(X_te_8))
    
    interp = "Low penalty -> Overfitting" if a <= 0.01 else ("Optimal Sweet Spot" if a <= 1.0 else "High penalty -> Underfitting")
    print(f"{a:<10.3f} {tr_e:<14.4f} {te_e:<14.4f} {interp}")